# Домашнє завдання до тем apply(), groupby(), pivot_table()

В цьому домашньому завданні продовжуємо працювати з набором даних `supermarket_sales.csv`.

0. Імпортуйте бібліотеку pandas та зчитайте дані у змінну `df` типу `pandas.DataFrame`.

In [1]:
import pandas as pd
df = pd.read_csv('../data/supermarket_sales.csv')

1. Дослідимо, який філіал супермаркету ('Branch') є найприбутковішим. Для цього знайдіть сумарний прибуток за кожним філіалом і виявіть, який філіал має найвищий.

In [2]:
s = df.groupby('Branch')['gross income'].sum().round(2)
top_branch = s.idxmax()
top_value = s.max()
print(f'Найприбутковіший філіал: {top_branch} - {top_value}')

Найприбутковіший філіал: C - 5265.18


2. В якому місті знаходиться філіал з найвищим прибутком? Може в тому місці нам розмітисти ще один магазин.  
Знайдіть відповідь за допомогою функціоналу Pandas.

In [3]:
top_city = df.loc[df['Branch'] == top_branch, 'City'].unique()[0]
print(f'Місто філіалу з найвищим прибутком: {top_city}')

Місто філіалу з найвищим прибутком: Naypyitaw


3.1. Створіть зведену таблицю, яка покаже, скільки покупок (інвойсів) було зроблено в кожній з філій (`Branch`) за різними категоріями товарів. Запишіть таблицю в змінну `invoices_by_category` і виведіть змінну на екран.
Ця таблиця допоможе проаналізувати, в якій філії купують найбільше товарів кожної з категорій.

In [4]:
invoices_by_category = pd.pivot_table(df, index='Branch', columns='Product line', values='Invoice ID', aggfunc='count')
invoices_by_category

Product line,Electronic accessories,Fashion accessories,Food and beverages,Health and beauty,Home and lifestyle,Sports and travel
Branch,,,,,,
A,60,51,58,47,65,59
B,55,62,50,53,50,62
C,55,65,66,52,45,45


Очікуваний результат:

![](https://drive.google.com/uc?export=view&id=1rueAdko6S3UxIHGtojetTxlES-EyM6Yb)

3.2. Викристовуючи змінну `invoices_by_category` дайте відповідь програмно (тобто значення треба не просто знайти очима, а вивести за допомогою коду), в якому філіалі магазину (`Branch`) найбільше інвойсів із покупкою товарів категорії "Електронні аксесуари" (`Electronic accessories`)?


In [5]:
invoices_by_category['Electronic accessories'].idxmax()

'A'

4-6. **Творче завдання на розвиток аналітичного мислення**

Крок 1. Сформулюйте ТРИ питання (гіпотези) до наявних даних, які допомогли б вам зрозуміти, які користувачі що, де та коли найбільше/найменше купують, аби дати на основі цих гіпотез рекомендації бізнесу. Звісно питання мають бути не тими, на які ми вже відповіли в завданнях модулю.

Крок 2. Знайдіть відповіді на свої питання з допомогою функціоналу pandas.

Крок 3. Напишіть, як відповідь на це питання може бути використана для прийняття бізнес рішень.   
   
 Питання можуть бути будь-якої складності, але їх має бути 3. Кожне питання оцінюється як 1 завдання. Без виконання цього завдання ДЗ не приймається. Якщо є питання щодо виконання - пишіть у чат 🙌


### 1. Який тип клієнта (Member / Normal) приносить найбільший сумарний прибуток, і чи є різниця між статями?
- **Member**: 7820.164 vs **Normal**: 7559.205 -> різниця **~260.959** (≈ **3.45%**).
- У **Member** більший внесок роблять **жінки**, а у **Normal** - **майже рівно** (дуже мала різниця).

**Рекомендація:** посилити програми лояльності/промо для сегмента, який приносить більше прибутку (Member), щоб утримувати й збільшувати частоту покупок.  
Додатково варто перевірити, за рахунок чого виникає різниця: більша кількість покупок чи вищий середній чек / прибуток на інвойс.


In [6]:
pd.pivot_table(df, index='Customer type', columns='Gender', values='gross income', aggfunc='sum')

Gender,Female,Male
Customer type,,
Member,4197.4735,3622.6905
Normal,3796.9515,3762.2535


### 2. Хто частіше робить покупки (Male/Female) і чи відрізняється “кошик” за категоріями товарів?

- Покупок майже порівну: **Female 501**, **Male 499**.
- За категоріями є різниця у структурі покупок:
  - жінки частіше: **Fashion accessories (19.2%)**, **Food and beverages (18.0%)**, **Sports and travel (17.6%)**
  - чоловіки частіше: **Health and beauty (17.6%)**, **Electronic accessories (17.2%)**, **Home and lifestyle (16.2%)**

**Як використати для бізнесу:** різниця між статями невелика, тому робити “жорсткі” рішення (тільки під стать) не потрібно.  
Натомість можна *тестувати промо/викладку* в категоріях, де різниця найбільша (наприклад, *Fashion accessories* vs *Health and beauty*).


In [7]:
# Хто частіше купує (кількість інвойсів)
df.Gender.value_counts()

Gender
Female    501
Male      499
Name: count, dtype: int64

In [8]:
# Розподіл покупок за категоріями (інвойси) для кожної статі
gender_product_mix = pd.pivot_table(df, index='Gender', columns='Product line', values='Invoice ID', aggfunc='count')
gender_product_mix

Product line,Electronic accessories,Fashion accessories,Food and beverages,Health and beauty,Home and lifestyle,Sports and travel
Gender,,,,,,
Female,84,96,90,64,79,88
Male,86,82,84,88,81,78


In [9]:
# Частки по рядку, щоб порівнювати структуру, а не абсолютні числа
gender_product_mix.div(gender_product_mix.sum(axis=1), axis=0).round(3)

Product line,Electronic accessories,Fashion accessories,Food and beverages,Health and beauty,Home and lifestyle,Sports and travel
Gender,,,,,,
Female,0.168,0.192,0.180,0.128,0.158,0.176
Male,0.172,0.164,0.168,0.176,0.162,0.156


### 3. Які категорії товарів є топовими в кожному філіалі (за обсягом продажів та прибутком) і чи збігаються ці категорії?

Категорії збігаються: у кожному філіалі лідер за обсягом продажів також є лідером за прибутком.

**Як це можна використати для бізнесу**

- У кожному філіалі є “своя” топ-категорія, яка дає і найбільший обсяг, і найбільший прибуток -> це основний драйвер.
- Тому варто тримати її в наявності (запаси/поставка) і робити кращу викладку саме для цієї категорії в кожному філіалі.
- **Промо краще робити по філіях**, під їхній топ (а не однаково для всіх) — так ефект буде більший.


In [10]:
pt = pd.pivot_table(df, index='Product line', columns='Branch', values=['Quantity', 'gross income'], aggfunc='sum')
pt

Quantity           gross income                     
Branch                        A    B    C            A         B          C
Product line                                                               
Electronic accessories      322  316  333     872.2435  811.9735   903.2845
Fashion accessories         263  297  342     777.7385  781.5865  1026.6700
Food and beverages          313  270  369     817.2905  724.5185  1131.7550
Health and beauty           257  320  277     599.8930  951.4600   791.2060
Home and lifestyle          371  295  245    1067.4855  835.6745   661.6930
Sports and travel           333  322  265     922.5095  951.8190   750.5680

In [11]:
qty = pt['Quantity']
inc = pt['gross income']

top_qty = qty.idxmax()   # топ категорія за обсягом для кожного Branch
top_inc = inc.idxmax()   # топ категорія за прибутком для кожного Branch

top_qty, top_inc, (top_qty == top_inc)

(Branch
 A    Home and lifestyle
 B     Sports and travel
 C    Food and beverages
 dtype: str,
 Branch
 A    Home and lifestyle
 B     Sports and travel
 C    Food and beverages
 dtype: str,
 Branch
 A    True
 B    True
 C    True
 dtype: bool)

**Time spent — 2 год**

**What I learned**
- **Групування та агрегації**: `groupby()` для підрахунків і порівняння показників між групами (Branch/Customer type/Gender).
- **Зведені таблиці**: `pivot_table()` для швидкого порівняння категорій між філіями/статями та роботи з кількома метриками.
- **Трансформації даних**: використання `apply()` для створення/перетворення колонок (коли потрібно застосувати функцію до значень).
- **Пошук “лідерів” у даних**: `max()` та `idxmax()` — знаходити не лише найбільше значення, а й групу/категорію, яка його дає.
- **Робота з фільтрацією**: `loc` для відбору рядків і діставання потрібних значень.
- **Аналітичне мислення**: формулювати гіпотези, перевіряти їх кодом і робити короткі висновки та рекомендації для бізнесу.